# 02 — Quantum-feature linear ridge: synthetic smoke check

This rerunnable prototype checks the package API, not molecular prediction quality or M4 completion.
It uses **synthetic, dimensionless inputs** and artificial labels; no QM9 test sets or IBM hardware are accessed.

The model fits a training-only scaler, encodes one input per qubit with Qiskit's `zz_feature_map`,
and computes local Z expectations with `StatevectorEstimator`. An expectation is the average result
of a quantum measurement, between -1 and 1. Classical `Ridge` learns the linear mapping to labels in debye.

Two circuit repetitions avoid the constant Z-only readout of a single phase-encoding layer.
The defaults are illustrative, not tuned. See `docs/PLAN.md` §7.9 and `tests/test_quantum.py`.

In [ ]:
try:
    from qm9dipole.provenance import check_environment
except ModuleNotFoundError as e:
    raise RuntimeError(
        'Install the project requirements and select its environment as the notebook kernel; see README.md.'
    ) from e

print(check_environment())

## 1. Exercise the regressor on small synthetic arrays

Four columns correspond to four qubits. The quantum circuit is fixed; only the scaler and
ridge coefficients are fitted. Scaling is refitted on every call to `fit`, including each CV fold.
The validation rows below are synthetic development examples, not the reserved molecular test sets.

In [ ]:
import numpy as np
import pandas as pd

from qm9dipole.models.quantum import QuantumRidgeRegressor

rng = np.random.default_rng(2026)
X_train = rng.normal(size=(24, 4))
X_validation = rng.normal(size=(4, 4))
y_train = 2.0 + 0.5 * np.sin(X_train[:, 0]) - 0.2 * X_train[:, 1]
y_validation = 2.0 + 0.5 * np.sin(X_validation[:, 0]) - 0.2 * X_validation[:, 1]

model = QuantumRidgeRegressor(alpha=1.0, gamma=1.0, reps=2)
model.fit(X_train, y_train)
model.get_params()

## 2. Inspect features and predictions

The features are dimensionless expectations. Predictions are in the same units as the labels,
and negative magnitude predictions are clipped to zero by default. No accuracy claim is made from this table.

In [ ]:
phi = model.quantum_features(X_validation)
predictions = model.predict(X_validation)
assert phi.shape == X_validation.shape
assert np.isfinite(phi).all() and np.isfinite(predictions).all()
assert np.all(np.abs(phi) <= 1.0 + 1e-12)

table = pd.DataFrame(
    phi, columns=model.pipeline_.named_steps['quantum'].get_feature_names_out()
)
table['synthetic_label_D'] = y_validation
table['prediction_D'] = predictions
table

In [ ]:
model.pipeline_.named_steps['quantum'].circuit_.draw(output='text')

## Before molecular benchmarking

- Supply a fixed-width descriptor table aligned by molecule ID and the corresponding debye labels.
- If PCA is learned, place it upstream of this regressor in the same scikit-learn `Pipeline` so
  it is fitted on each training/CV fold; do not cross-validate globally fitted PCA scores.
- Use invariant molecular descriptors: encoding flattened XYZ does not create rotation, translation or atom-order invariance.
- Compare matched classical and quantum models on identical training subsets/folds and tune only on training data.
- Freeze settings before the separate final test evaluation.

This module performs exact local simulation only. Finite shots/noise and a separately approved hardware
experiment are future work. At 4–8 qubits, classical simulation is cheap; this prototype makes no speedup claim.